## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI:

```bash
az login
```

# 🏦 Custom Chat Message Store for Threads

## Industry Use Case: Compliance-Ready Conversation Audit Trail

This notebook demonstrates how to implement a **custom chat message store** for managing conversation threads with compliance and audit requirements.

| Feature | FSI Application |
|---------|----------------|
| **Custom Storage** | Store conversations in compliance-approved databases |
| **Full Serialization** | Complete message history for regulatory audits |
| **Data Control** | Meet data residency and retention requirements |
| **Flexible Backend** | Integrate with existing enterprise systems |

In [1]:
import os
from typing import Any, Sequence
from azure.identity import AzureCliCredential
from agent_framework import Agent, AgentSession, Message, HistoryProvider
from agent_framework_foundry import FoundryChatClient
from dotenv import load_dotenv

load_dotenv("../../.env")
print("✅ Environment loaded and dependencies imported")

<frozen abc>:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
<frozen abc>:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.


✅ Environment loaded and dependencies imported


## Implement Custom Chat Message Store

This store could integrate with compliance-approved databases (PostgreSQL, Cosmos DB, etc.) for regulatory audit requirements.

In [2]:
class CustomHistoryProvider(HistoryProvider):
    """Custom history provider for compliance-ready conversation storage.
    
    In production, this would integrate with:
    - Relational databases (PostgreSQL, SQL Server)
    - NoSQL databases (Cosmos DB, MongoDB)
    - Enterprise storage systems with audit capabilities
    """

    def __init__(self, source_id: str = "compliance-history") -> None:
        super().__init__(source_id)
        # Store messages per session_id
        self._sessions: dict[str, list[Message]] = {}

    async def get_messages(
        self, session_id: str | None, *, state: dict[str, Any] | None = None, **kwargs: Any
    ) -> list[Message]:
        """Retrieve all messages for a session (would query database in production)."""
        if session_id is None:
            return []
        return list(self._sessions.get(session_id, []))

    async def save_messages(
        self,
        session_id: str | None,
        messages: Sequence[Message],
        *,
        state: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> None:
        """Save messages for a session (would write to database in production)."""
        if session_id is None:
            return
        if session_id not in self._sessions:
            self._sessions[session_id] = []
        self._sessions[session_id].extend(messages)

print("✅ CustomHistoryProvider defined")

✅ CustomHistoryProvider defined


## Define Compliance Officer Tools

In [3]:
async def check_transaction_compliance(transaction_id: str) -> str:
    """Check if a transaction meets compliance requirements."""
    compliance_data = {
        "TXN-001": {"status": "Compliant", "risk_level": "Low", "aml_check": "Passed"},
        "TXN-002": {"status": "Under Review", "risk_level": "High", "aml_check": "Pending"},
        "TXN-003": {"status": "Flagged", "risk_level": "Critical", "aml_check": "Failed"},
    }
    c = compliance_data.get(transaction_id, {"status": "Unknown", "risk_level": "Unknown", "aml_check": "Unknown"})
    return f"Transaction {transaction_id}: Status={c['status']}, Risk={c['risk_level']}, AML Check={c['aml_check']}"

async def get_regulatory_requirements(regulation_type: str) -> str:
    """Get regulatory requirements for a specific regulation."""
    regulations = {
        "AML": "Anti-Money Laundering: Customer verification, transaction monitoring, suspicious activity reporting",
        "KYC": "Know Your Customer: Identity verification, risk assessment, ongoing monitoring",
        "GDPR": "Data Protection: Consent management, data minimization, right to erasure",
    }
    return regulations.get(regulation_type, f"No specific requirements found for {regulation_type}")

print("✅ Tools defined: check_transaction_compliance, get_regulatory_requirements")

✅ Tools defined: check_transaction_compliance, get_regulatory_requirements


## Run Compliance Officer with Custom Store

The agent uses `chat_message_store_factory` to create custom store instances for each thread.

In [4]:
async def run_compliance_officer():
    """Run compliance officer with custom history provider."""
    print("\n--- Compliance Officer with Custom History Provider ---\n")
    
    # FoundryChatClient uses the Foundry Responses API with local history management
    endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")
    
    print(f"Using Foundry endpoint: {endpoint}")
    print(f"Using deployment: {deployment_name}\n")
    
    # Create a shared history provider instance
    history_provider = CustomHistoryProvider("compliance-history")
    
    agent = FoundryChatClient(
        project_endpoint=endpoint,
        model=deployment_name,
        credential=AzureCliCredential(),
    ).as_agent(
        name="ComplianceOfficer",
        instructions="You are a compliance officer helping with regulatory inquiries. Check transactions and provide regulatory guidance.",
        tools=[check_transaction_compliance, get_regulatory_requirements],
        context_providers=[history_provider],  # Use custom history provider
    )
    
    # Create a new session
    session = agent.create_session()
    print(f"✅ Session created: {session.session_id}\n")
    
    # Phase 1: Initial conversation
    print("=" * 60)
    print("Phase 1: Compliance Review Session")
    print("=" * 60)
    
    query1 = "Please check compliance status for transaction TXN-002"
    print(f"Analyst: {query1}")
    response1 = await agent.run(query1, session=session)
    print(f"Officer: {response1.text}\n")
    
    # Phase 2: Serialize session
    print("=" * 60)
    print("Phase 2: Saving Compliance Session")
    print("=" * 60)
    
    serialized_session = session.to_dict()
    
    print(f"\n💾 Serialized session: {serialized_session}\n")
    print("📊 Note: History provider stores FULL message history per session")
    print("💡 This enables complete audit trails for compliance\n")
    
    # Phase 3: Resume session
    print("=" * 60)
    print("Phase 3: Resuming Compliance Session")
    print("=" * 60)
    
    resumed_session = AgentSession.from_dict(serialized_session)
    print(f"\n✅ Session restored: {resumed_session.session_id}\n")
    
    query2 = "Based on our previous conversation, what transaction did we discuss?"
    print(f"Analyst: {query2}")
    response2 = await agent.run(query2, session=resumed_session)
    print(f"Officer: {response2.text}\n")
    
    print("=" * 60)
    print("✅ Custom history provider demo completed!")
    print("=" * 60)

await run_compliance_officer()


--- Compliance Officer with Custom History Provider ---

Using Foundry endpoint: https://kd-foundry.services.ai.azure.com/api/projects/proj-kd
Using deployment: gpt-4o



✅ Session created: a5ce36f9-401d-4819-a914-892d7a0918a3

Phase 1: Compliance Review Session
Analyst: Please check compliance status for transaction TXN-002


Officer: Transaction TXN-002 is currently under review with a high risk designation. The AML check is still pending. Please ensure all necessary documentation and information are provided for further assessment.

Phase 2: Saving Compliance Session

💾 Serialized session: {'type': 'session', 'session_id': 'a5ce36f9-401d-4819-a914-892d7a0918a3', 'service_session_id': 'resp_03493350bd58aac1006a01d5c402f08194886f3f3edb8a2ec2', 'state': {'compliance-history': {}}}

📊 Note: History provider stores FULL message history per session
💡 This enables complete audit trails for compliance

Phase 3: Resuming Compliance Session

✅ Session restored: a5ce36f9-401d-4819-a914-892d7a0918a3

Analyst: Based on our previous conversation, what transaction did we discuss?


Officer: We discussed transaction TXN-002. It was noted to be under review with a high risk designation, and the AML check was pending.

✅ Custom history provider demo completed!


## Key Takeaways

| Feature | Description |
|---------|-------------|
| `ChatMessageStoreProtocol` | Interface for custom storage backends |
| `chat_message_store_factory` | Factory to create store instances per thread |
| `serialize()` / `deserialize()` | Store state for compliance audit trails |

## Production Considerations

| Aspect | Implementation |
|--------|---------------|
| **Database** | PostgreSQL, Cosmos DB, SQL Server |
| **Audit Trail** | Immutable logs with timestamps |
| **Encryption** | Encrypt sensitive conversation data |
| **Retention** | Implement data retention policies |